# Lab 04 — EDA — TV, Radio, Newspaper vs Sales
**First Real EDA Track** · Beginner · ~45 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Load data with pandas and summarise with describe()
2. Compute and interpret a correlation matrix
3. Create scatter plots and quartile-binned means
4. Write a 5-bullet findings memo with design caveats

## Datasets (this folder)
- `Advertising.csv` — auto-download from `https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/master/data/Advertising.csv`

## How to run on Google Colab
1. Click **Start Lab** — the hosted notebook opens directly in Colab under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0 (bootstrap)** first — it pulls `dataset.zip` from the lab manifest into `/content/ml_lab` (falls back to public raw URLs, then local files).
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 fetches `dataset.zip` from the manifest → `Runtime → Run all`.


### Setup (VLABS bootstrap)

Run the next cell (Cell 0) once. Fetch order: hosted `manifest.json` → `dataset.zip` extracted to `/content/ml_lab/<lab_id>` → per-file public raw URLs → local files next to this notebook. No-op when files already exist.


In [ ]:
# Cell 0 — VLABS bootstrap: run first. Works on Colab (direct-open URL) and locally.
import io, json, os, urllib.request, zipfile

LAB_ID = "lab-04-eda-advertising"
# Hosted manifest (Admin: replace ORG/REPO once per deployment).
MANIFEST_URL = f"https://raw.githubusercontent.com/ORG/REPO/main/{LAB_ID}/manifest.json"
# Alternative: backend proxy to S3 — uncomment to use instead:
# MANIFEST_URL = f"https://api.vlabs.test/colab/{LAB_ID}/manifest"
ON_COLAB = os.path.isdir("/content")
DATA_DIR = f"/content/ml_lab/{LAB_ID}" if ON_COLAB else "."

def _fetch(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read()

def _ensure_file(filename, url=None):
    """Local-first single-file fetch (also used by lesson load cells)."""
    for base in (DATA_DIR, "."):
        p = os.path.join(base, filename)
        if os.path.exists(p):
            print(f"found {p}")
            return p
    if not url:
        raise FileNotFoundError(
            f"{filename} missing: open via Start Lab (bundle) or add it next to the notebook")
    os.makedirs(DATA_DIR, exist_ok=True)
    dest = os.path.join(DATA_DIR, filename)
    print(f"downloading {filename} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"saved {dest}")
    return dest

ensure = _ensure_file  # compat alias for lesson load cells

def vlabs_bootstrap():
    # 1) Hosted manifest -> dataset.zip -> DATA_DIR (direct-open path)
    try:
        m = json.loads(_fetch(MANIFEST_URL).decode("utf-8"))
        dz = m.get("dataset_zip")
        if dz:
            print(f"manifest ok: {MANIFEST_URL}")
            os.makedirs(DATA_DIR, exist_ok=True)
            zpath = os.path.join(DATA_DIR, "dataset.zip")
            urllib.request.urlretrieve(dz, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(DATA_DIR)
            print(f"extracted dataset.zip -> {DATA_DIR}")
    except Exception as e:
        print(f"manifest skip ({e}); using file fallbacks")
    # 2) Per-file fallbacks (public raw URLs; local files are a no-op hit)
    _ensure_file("Advertising.csv", "https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/master/data/Advertising.csv")
    # 3) Work from the data dir on Colab so relative paths resolve
    if ON_COLAB and DATA_DIR != ".":
        os.chdir(DATA_DIR)
        print(f"cwd -> {DATA_DIR}")

vlabs_bootstrap()


## First Real EDA Track: Correlation, Plots, Findings Memo

> **Scenario:** Marketing asks *which channel drives Sales*. Load `Advertising.csv` (200 rows), compute correlations, plot scatters, bin TV into quartiles, and write a 5-bullet findings memo.
>
> **You will learn:** pandas `read_csv` / `describe` / `corr`, matplotlib scatter, correlation vs causation.
> **Time:** ~45 minutes. **Level:** Beginner. **Needs:** pandas + matplotlib. **Env:** 🟢 Colab only.

### EDA mental map

| Question | Statistic / view | Code |
|---|---|---|
| How big / spread? | `describe()` | `df.describe()` |
| What moves with Sales? | Pearson r | `df.corr()["Sales"]` |
| Linear relationship? | scatter | `plt.scatter(df["TV"], df["Sales"])` |
| Dose–response? | binned means | `pd.qcut` + `groupby.mean` |

---

### 1. Load data (local first, Colab fallback)

In [ ]:
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe; drop in interactive notebooks
import matplotlib.pyplot as plt

def load_ads():
    local = "Advertising.csv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/master/data/Advertising.csv",
            local,
        )
    df = pd.read_csv(local)
    # First column is an unnamed index
    if str(df.columns[0]).startswith("Unnamed"):
        df = df.rename(columns={df.columns[0]: "index"})
    return df

df = load_ads()
print(df.shape)   # (200, 5)
print(df.head(3))
print(df.describe().round(2))


Expected highlights: `Sales` mean ≈ 14.02, min 1.6, max 27.0; `TV` mean ≈ 147.04.

---

### 2. Correlation matrix

In [ ]:
corr = df[["TV", "Radio", "Newspaper", "Sales"]].corr().round(4)
print(corr)


Expected `Sales` column:

| Feature | r with Sales |
|---|---|
| TV | **0.7822** |
| Radio | 0.5762 |
| Newspaper | 0.2283 |

In [ ]:
print("strongest single-feature |r| with Sales:",
      corr["Sales"].drop("Sales").abs().idxmax())  # TV


> Correlation ≠ causation: this is observational spend data, not a randomized test. Say “associated with”, not “drives”, in the memo.

---

### 3. Scatter matrix (three panels)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, col in zip(axes, ["TV", "Radio", "Newspaper"]):
    ax.scatter(df[col], df["Sales"], alpha=0.6, edgecolors="none")
    ax.set_xlabel(col)
    ax.set_ylabel("Sales")
    ax.set_title(f"{col} vs Sales  r={df[col].corr(df['Sales']):.3f}")
fig.tight_layout()
fig.savefig("eda_scatters.png", dpi=120)
print("saved eda_scatters.png")


---

### 4. TV quartiles → mean Sales

In [ ]:
df["TV_bin"] = pd.qcut(df["TV"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
tv_bins = df.groupby("TV_bin", observed=True)["Sales"].agg(["mean", "count"]).round(3)
print(tv_bins)


Expected (edges ≈ 74.38 / 149.75 / 218.83):

| TV_bin | mean Sales | n |
|---|---|---|
| Q1 | 8.390 | 50 |
| Q2 | 12.686 | 50 |
| Q3 | 16.548 | 50 |
| Q4 | 18.466 | 50 |

Monotone increase with TV spend — stronger evidence than raw r alone.

In [ ]:
ax = tv_bins["mean"].plot(kind="bar", figsize=(6, 3), rot=0,
                          title="Mean Sales by TV quartile")
ax.set_ylabel("Mean Sales")
fig = ax.get_figure(); fig.tight_layout(); fig.savefig("eda_tv_quartiles.png", dpi=120)


---

### 5. Five-bullet findings memo

Write `labs/eda_findings.md` (or cell markdown) with bullets like:

1. **TV is the strongest linear associate of Sales** (r ≈ 0.78), ahead of Radio (0.58) and Newspaper (0.23).
2. **Mean Sales rises monotonically across TV quartiles** (8.4 → 18.5), so the relationship is not driven by a few outliers alone.
3. **Newspaper’s weak r (0.23) largely tracks Radio** (r(Newspaper, Radio) ≈ 0.35); partial correlation / regression would likely shrink Newspaper’s unique contribution.
4. **TV and Radio spend are nearly uncorrelated** (r ≈ 0.05) — channels can be evaluated independently in this sample.
5. **Design limit:** observational data, n = 200, one market — treat findings as hypotheses for a geo test, not proof of causality.

In [ ]:
memo = """# EDA findings — Advertising channels
- TV strongest correlate of Sales (r=0.78).
- Mean Sales rises monotonically by TV quartile (8.4→18.5).
- Newspaper weak alone (r=0.23); overlaps Radio.
- TV vs Radio spend ~uncorrelated (r=0.05).
- Observational, n=200: associations only, not causal proof.
"""
open("eda_findings.md", "w", encoding="utf-8").write(memo)
print("wrote eda_findings.md")


---

## Exercises (do these!)

### Exercise 1 — Full correlation matrix
Print `df[["TV","Radio","Newspaper","Sales"]].corr()` rounded to 4 d.p. Which pair of *features* (not Sales) is most correlated?
*Expected: Radio–Newspaper ≈ 0.3541 is the strongest feature–feature pair (TV pairs are ≈ 0.05).*

<details>
<summary>Hint</summary>

```python
c = df[["TV", "Radio", "Newspaper"]].corr()
print(c)
# upper triangle only to avoid duplicates
```

</details>

### Exercise 2 — Highest single-feature R vs Sales
Using absolute correlation with `Sales`, which feature wins and what is r?
*Expected: TV, r ≈ 0.7822.*

<details>
<summary>Hint</summary>

`df.corr()["Sales"].drop("Sales").abs().idxmax()` / `.max()`.
</details>

### Exercise 3 — TV quartiles → mean Sales
Bin `TV` into 4 equal-count bins (`pd.qcut(..., 4)`), print mean `Sales` per bin.
*Expected: Q1 8.39 · Q2 12.686 · Q3 16.548 · Q4 18.466 (50 rows each).*

<details>
<summary>Hint</summary>

`pd.qcut(df["TV"], 4, labels=["Q1","Q2","Q3","Q4"])` then `groupby(...).mean()`.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
print(df[["TV", "Radio", "Newspaper", "Sales"]].corr().round(4))
# Radio-Newspaper ≈ 0.3541 is the strongest non-Sales pair.

# --- Solution 2 ---
s = df.corr(numeric_only=True)["Sales"].drop("Sales")
top = s.abs().idxmax()
print(top, round(s[top], 4))  # TV 0.7822

# --- Solution 3 ---
df["TV_bin"] = pd.qcut(df["TV"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
print(df.groupby("TV_bin", observed=True)["Sales"].mean().round(3))
# Q1  8.390
# Q2 12.686
# Q3 16.548
# Q4 18.466


### What to learn next
- Simple OLS: `sm.OLS(Sales, add_constant(df[["TV","Radio"]])).fit()` — does Newspaper add anything?
- Seaborn `pairplot(df, hue=None)` for a one-line scatter matrix.
- Then train/test split before trusting fit metrics (Course 2).
- Cheat sheet: describe → corr → scatter → binned means → memo; always state n and design limits.

*Files in this folder: `Advertising.csv` · outputs `eda_scatters.png`, `eda_tv_quartiles.png`, `eda_findings.md` when you run the lab.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
